In [ ]:
import os
import json

In [ ]:
files_data = []
for root, dirs, files in os.walk("./_result"):
    for file in files:
        if file.startswith("_result_day_per_timept_07_22") and file.endswith(".json"):
        # if file.startswith("_result_day_s40") and file.endswith(".json"):
            file_path = os.path.join(root, file)
            files_data.append(json.loads(open(file_path, "r").read()))

In [ ]:
import csv
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font

ATTR_ORDER  = ['Temp', 'RH', 'WSpd', 'WDir']
MODEL_ORDER = ['LSTM', 'LSTM-FC', 'LSTM-ED', 'LSTM-ED-FC', 'LSTM-ED-ATTN-FC', 'LSTM-ED-CNN-FC']

METRIC_COLS = {
    3: False, 4: False, 5: True,
    6: False, 7: False, 8: True,
    9: False, 10: False, 11: True,
}

def compile_full_results(files_data, out_path='_result/compiled_full_per_timestep_stnsAvg.csv_07_22', excel_path='_result/compiled_full_per_timestep_stnsAvg_07_22.xlsx'):
    acc = {}

    for file_entry in files_data:
        for model, attrs in file_entry.items():
            if not isinstance(attrs, dict):
                continue
            for attr, splits in attrs.items():
                if not isinstance(splits, dict):
                    continue
                for split in ('train', 'eval', 'test'):
                    split_data = splits.get(split, {})
                    # Average across all station keys (exclude 'full')
                    entries = [v for k, v in split_data['stations'].items() if isinstance(v, dict)]
                    print(list(split_data.items()))
                    mae_vals   = [e['mae']   for e in entries if 'mae'   in e]
                    rmse_vals  = [e['rmse']  for e in entries if 'rmse'  in e]
                    r2_vals    = [e['r2']    for e in entries if 'r2'    in e]
                    count_vals = [e['count'] for e in entries if 'count' in e]
                    if not mae_vals:
                        continue
                    n = len(mae_vals)
                    key = (model, attr, split)
                    if key not in acc:
                        acc[key] = {'mae': [], 'rmse': [], 'r2': [], 'count': []}
                    acc[key]['mae'].append(sum(mae_vals) / n)
                    acc[key]['rmse'].append(sum(rmse_vals) / n)
                    acc[key]['r2'].append(sum(r2_vals) / n)
                    acc[key]['count'].append(int(sum(count_vals) / n) if count_vals else 0)

    rows = []
    for (model, attr, split), vals in sorted(acc.items()):
        n = len(vals['mae'])
        rows.append({
            'model': model, 'attr': attr, 'split': split,
            'mae':    sum(vals['mae'])   / n,
            'rmse':   sum(vals['rmse'])  / n,
            'r2':     sum(vals['r2'])    / n,
            'count':  int(sum(vals['count']) / n),
            'n_files': n,
        })

    # with open(out_path, 'w', newline='') as f:
    #     writer = csv.DictWriter(f, fieldnames=['model', 'attr', 'split', 'mae', 'rmse', 'r2', 'count', 'n_files'])
    #     writer.writeheader()
    #     writer.writerows(rows)
    # print(f'Written {len(rows)} rows to {out_path}')

    pivot = {}
    for row in rows:
        key = (row['attr'], row['model'])
        if key not in pivot:
            pivot[key] = {'model': row['model'], 'attr': row['attr']}
        s = row['split']
        pivot[key][f'{s}-MAE']  = row['mae']
        pivot[key][f'{s}-RMSE'] = row['rmse']
        pivot[key][f'{s}-R2']   = row['r2']

    def sort_key(entry):
        attr, model = entry
        ai = ATTR_ORDER.index(attr)   if attr  in ATTR_ORDER  else len(ATTR_ORDER)
        mi = MODEL_ORDER.index(model) if model in MODEL_ORDER else len(MODEL_ORDER)
        return (ai, mi)

    sorted_rows = [pivot[k] for k in sorted(pivot.keys(), key=sort_key)]

    wb = Workbook()
    ws = wb.active

    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    bold   = Font(bold=True)

    for col_letter, label in [('A', 'model'), ('B', 'attr'),
                               ('C', 'Train'), ('F', 'Eval'), ('I', 'Test')]:
        ws[f'{col_letter}1'] = label
        ws[f'{col_letter}1'].alignment = center
        ws[f'{col_letter}1'].font = bold

    ws.merge_cells('A1:A2')
    ws.merge_cells('B1:B2')
    ws.merge_cells('C1:E1')
    ws.merge_cells('F1:H1')
    ws.merge_cells('I1:K1')

    for col, header in enumerate(['MAE', 'RMSE', 'R2',
                                   'MAE', 'RMSE', 'R2',
                                   'MAE', 'RMSE', 'R2'], start=3):
        cell = ws.cell(2, col, header)
        cell.alignment = center
        cell.font = bold

    DATA_START = 3
    for r_idx, row in enumerate(sorted_rows, start=DATA_START):
        ws.cell(r_idx, 1, row['model'])
        ws.cell(r_idx, 2, row['attr'])
        ws.cell(r_idx, 3, row.get('train-MAE'))
        ws.cell(r_idx, 4, row.get('train-RMSE'))
        ws.cell(r_idx, 5, row.get('train-R2'))
        ws.cell(r_idx, 6, row.get('eval-MAE'))
        ws.cell(r_idx, 7, row.get('eval-RMSE'))
        ws.cell(r_idx, 8, row.get('eval-R2'))
        ws.cell(r_idx, 9, row.get('test-MAE'))
        ws.cell(r_idx, 10, row.get('test-RMSE'))
        ws.cell(r_idx, 11, row.get('test-R2'))

    attr_row_ranges = {}
    for i, row in enumerate(sorted_rows):
        attr = row['attr']
        excel_row = DATA_START + i
        if attr not in attr_row_ranges:
            attr_row_ranges[attr] = []
        attr_row_ranges[attr].append(excel_row)

    for attr, row_indices in attr_row_ranges.items():
        for col, higher_is_better in METRIC_COLS.items():
            candidates = [(ws.cell(r, col).value, r) for r in row_indices if ws.cell(r, col).value is not None]
            if not candidates:
                continue
            _, best_row = max(candidates) if higher_is_better else min(candidates)
            ws.cell(best_row, col).font = bold

    wb.save(excel_path)
    print(f'Written {len(sorted_rows)} rows to {excel_path}')

    return rows

rows = compile_full_results(files_data)
# for r in rows:
#     print(r)